# Importación de librerías

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import math
import seaborn as sns
import matplotlib.pyplot as plt

# Carga de datos

In [ ]:
RUTA_DATOS = Path("data") / "KaggleV2-May-2016.csv"

assert RUTA_DATOS.exists(), f"No se encontró el archivo: {RUTA_DATOS}"

df = pd.read_csv(RUTA_DATOS)

print("Dataset cargado.")
print("Dimensiones", df.shape)

Dataset cargado.
Dimensiones (110527, 14)


# Perfilamiento de calidad

In [ ]:
display(df.head(3))

df.info()

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


* Vemos que la columna PatientId usa float, lo cual es incorrecto al ser números enteros.
* Vemos que ScheduledDay y AppointmentDay son variables object (texto), deberían ser datetime.

In [ ]:
faltantes = pd.DataFrame({
    "cantidad": df.isna().sum(),
    "porcentaje": (df.isna().mean() * 100).round(2)
})

display(faltantes)

,cantidad,porcentaje
PatientId,0,0.0
AppointmentID,0,0.0
Gender,0,0.0
ScheduledDay,0,0.0
AppointmentDay,0,0.0
Age,0,0.0
Neighbourhood,0,0.0
Scholarship,0,0.0
Hipertension,0,0.0
Diabetes,0,0.0


In [ ]:
duplicados = df.duplicated().sum()

print("Cantidad de filas duplicadas:", duplicados)
print("Porcentaje de duplicados:", round(duplicados / len(df) * 100, 2))

Cantidad de filas duplicadas: 0
Porcentaje de duplicados: 0.0


In [ ]:
df.nunique()

,0
PatientId,62299
AppointmentID,110527
Gender,2
ScheduledDay,103549
AppointmentDay,27
Age,104
Neighbourhood,81
Scholarship,2
Hipertension,2
Diabetes,2


Validamos que la cantidad de AppointmentID coincide exactamente con el número de filas del dataset, esto implica que es una buena posible clave primaria.
Podemos destacar que existen 81 valores únicos para Neighbourhood.
También que hay una gran diferencia entre los valores únicos de Scheduled y Appointment, y 5 valores únicos para Handcap lo que nos dice que es una escala.

## Posibles outliers

In [ ]:
df.describe()

,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,1.105270e+05,1.105270e+05,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000,110527.000000
mean,1.474963e+14,5.675305e+06,37.088874,0.098266,0.197246,0.071865,0.030400,0.022248,0.321026
std,2.560949e+14,7.129575e+04,23.110205,0.297675,0.397921,0.258265,0.171686,0.161543,0.466873
min,3.921784e+04,5.030230e+06,-1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.172614e+12,5.640286e+06,18.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,3.173184e+13,5.680573e+06,37.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,9.439172e+13,5.725524e+06,55.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.999816e+14,5.790484e+06,115.000000,1.000000,1.000000,1.000000,1.000000,4.000000,1.000000


In [ ]:
edades_sospechosas = df[
    (df["Age"] < 0) | (df["Age"] > 110)
]
print(f"\nEDADES SOSPECHOSAS:")
display(edades_sospechosas)


EDADES SOSPECHOSAS:


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
63912,3.196321e+13,5700278,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
63915,3.196321e+13,5700279,F,2016-05-16T09:17:44Z,2016-05-19T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
68127,3.196321e+13,5562812,F,2016-04-08T14:29:17Z,2016-05-16T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,Yes
76284,3.196321e+13,5744037,F,2016-05-30T09:44:51Z,2016-05-30T00:00:00Z,115,ANDORINHAS,0,0,0,0,1,0,No
97666,7.482346e+14,5717451,F,2016-05-19T07:57:56Z,2016-06-03T00:00:00Z,115,SÃO JOSÉ,0,1,0,0,0,1,No
99832,4.659432e+14,5775010,F,2016-06-06T08:58:13Z,2016-06-06T00:00:00Z,-1,ROMÃO,0,0,0,0,0,0,No


## Coherencia temporal

In [ ]:
df["ScheduledDay"] = pd.to_datetime(
    df["ScheduledDay"],
    errors="coerce"
)

df["AppointmentDay"] = pd.to_datetime(
    df["AppointmentDay"],
    errors="coerce"
)

print("Fechas inválidas en ScheduledDay:",
      df["ScheduledDay"].isna().sum())

print("Fechas inválidas en AppointmentDay:",
      df["AppointmentDay"].isna().sum())

# Citas cuya fecha programada es anterior a la fecha de agendamiento
fechas_incoherentes = df[
    df["AppointmentDay"] < df["ScheduledDay"]
]

display(fechas_incoherentes)

Fechas inválidas en ScheduledDay: 0
Fechas inválidas en AppointmentDay: 0


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110511,8.235996e+11,5786742,F,2016-06-08 08:50:20+00:00,2016-06-08 00:00:00+00:00,14,MARIA ORTIZ,0,0,0,0,0,0,No
110512,9.876246e+13,5786368,F,2016-06-08 08:20:01+00:00,2016-06-08 00:00:00+00:00,41,MARIA ORTIZ,0,0,0,0,0,0,No
110513,8.674778e+13,5785964,M,2016-06-08 07:52:55+00:00,2016-06-08 00:00:00+00:00,2,ANTÔNIO HONÓRIO,0,0,0,0,0,0,No
110514,2.695685e+12,5786567,F,2016-06-08 08:35:31+00:00,2016-06-08 00:00:00+00:00,58,MARIA ORTIZ,0,0,0,0,0,0,No


La conversión de fechas fue hecha para poder analizarlas. A la vez vemos que 38568 AppointmentDay son menores que los ScheduledDay, notamos que esto es debido a que los datos de AppointmentDay no tienen hora, asi que para hacer una comparación fidedigna utilizaremos `.normalize()`

In [ ]:
fechas_incoherentes = df[
    df["AppointmentDay"].dt.normalize() < df["ScheduledDay"].dt.normalize()
]

print("Filas realmente incoherentes:", len(fechas_incoherentes))
display(fechas_incoherentes)

Filas realmente incoherentes: 5


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
27033,7.839273e+12,5679978,M,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,38,RESISTÊNCIA,0,0,0,0,1,0,Yes
55226,7.896294e+12,5715660,F,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,19,SANTO ANTÔNIO,0,0,0,0,1,0,Yes
64175,2.425226e+13,5664962,F,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,22,CONSOLAÇÃO,0,0,0,0,0,0,Yes
71533,9.982316e+14,5686628,F,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,81,SANTO ANTÔNIO,0,0,0,0,0,0,Yes
72362,3.787482e+12,5655637,M,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,7,TABUAZEIRO,0,0,0,0,0,0,Yes


## Revisión de variable objetivo

In [ ]:
print("Conteo:")
print(df["No-show"].value_counts(dropna=False))

print(f"\nProporciones:")
print(
    df["No-show"].value_counts(normalize=True, dropna=False) * 100
)

Conteo:
No-show
No     88208
Yes    22319
Name: count, dtype: int64

Proporciones:
No-show
No     79.806744
Yes    20.193256
Name: proportion, dtype: float64


## Resumen de calidad y hallazgos

A continuación se detallan los hallazgos principales obtenidos durante la fase de perfilamiento y análisis de calidad de datos:

* **Valores Faltantes y Duplicados:**
  * No se detectaron valores nulos (`NaN`) en ninguna de las columnas del dataset.
  * No existen filas completamente duplicadas dentro del conjunto de datos.

* **Inconsistencias en Tipos de Datos:**
  * La columna `PatientId` está registrada como tipo flotante (`float64`), cuando lo ideal para un identificador único es que sea entero (`int64`) o de tipo texto (`str`).
  * Las columnas `ScheduledDay` y `AppointmentDay` se encontraban inicialmente como texto (`object`), fue necesario convertirlas a formato fecha (`datetime64`) para su correcto análisis de calidad.

* **Valores únicos:**
  * AppointmentID como clave primaria válida (sin duplicados).
  * Neighbourhood con 81 valores únicos
  * Scheduled vs. Appointment: Gran diferencia en valores únicos.
  * Handcap como una escala de 5 valores únicos.

* **Inconsistencia de Fechas (Coherencia Temporal):**
  * Se identificaron **5 registros** donde la fecha de la cita (`AppointmentDay`) normalizada es estrictamente anterior a la fecha de agendamiento (`ScheduledDay`) normalizada, lo cual representa una anomalía temporal.

* **Valores Atípicos en la Edad (`Age`):**
  * Se detectaron **6 registros** con edades atípicas o dudosas fuera del rango esperado (edades negativas como `-1` o personas con edades extremadamente avanzadas que requieren validación, como `115` años).

* **Distribución de la Variable Objetivo (`No-show`):**
  * El dataset presenta un desbalance de clases: aproximadamente el **79.81%** de los pacientes sí asistieron a su cita (`No`), mientras que el **20.19%** faltaron (`Yes`).